# 第 6 章 · 模型组装(精简版)

> 本文是 [ch06.ipynb](./ch06.ipynb) 的浓缩版。只保留核心结构,方便快速复习。

## 完整 forward pass 的 shape 流水线

```
input_ids  (b, T)
    │ embed_tokens (查表)
    ▼
hidden     (b, T, 768)
    │ 8× MiniMindBlock
    ▼
hidden     (b, T, 768)     ← shape 不变,内容变了
    │ final RMSNorm
    ▼
hidden     (b, T, 768)
    │ lm_head (Linear 768→6400)
    ▼
logits     (b, T, 6400)
    │ shift + CE loss
    ▼
loss       scalar
```

## 两层类结构

| 类 | 行号 | 职责 |
|---|---|---|
| `MiniMindModel` | 210-264 | backbone:embed + 8×Block + norm + RoPE buffer |
| `MiniMindForCausalLM` | 266-310 | + lm_head + CE loss + tied weights |

## 关键设计

```python
# 1. 权重绑定:embed_tokens 和 lm_head 共享权重
_tied_weights_keys = {"lm_head.weight": "model.embed_tokens.weight"}
self.model.embed_tokens.weight = self.lm_head.weight  # 同一对象

# 2. start_pos 从 KV cache 推算
start_pos = past_key_values[0][0].shape[1] if past_key_values[0] is not None else 0
position_embeddings = (freqs_cos[start_pos:start_pos+seq_len], freqs_sin[...])

# 3. CE loss with -100 masking
loss = F.cross_entropy(logits[:, :-1], labels[:, 1:], ignore_index=-100)

# 4. MoE aux_loss 求和(密集模型为 0)
aux_loss = sum([l.mlp.aux_loss for l in self.layers if isinstance(l.mlp, MOEFeedForward)], 0)
```

## 参数量(63.9M)

| 组件 | 参数量 | 占比 |
|---|---|---|
| embed_tokens | 4.9M | 7.7% |
| 8× Attention | 14.2M | 22.2% |
| 8× FeedForward | 44.8M | 70.1% |
| Norms | 0.01M | 0.0% |
| **总计(tied)** | **63.9M** | **100%** |

> 不绑定(tied=False）: +4.9M = 68.8M

## 下一步

有了 logits → 怎么选下一个 token？→ **第 7 章:生成**